In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 2: Empty cells. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_empty as E, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The seven Empty-cells statement types (rules in scripts/edop/dashboard/wo02_rules_empty.py)
for s in E.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print(f"engine's own outside_active_domain threshold on weight_at_zero: {E.ENGINE_GUARD}")


EMP-01  Zero here, no percentile (one basin)
      applies to: single basin and one-basin buffer; area_weighted; L6, L8
      template  : {name} is zero in this basin. No percentile is given: {zf} of all L{level} basins are zero for this variable.
EMP-02  Zero across most of an area, no percentile
      applies to: buffer, polity; area_weighted; L6, L8
      template  : {name} is zero across {share} of this {area}. No percentile is given: {zf} of all L{level} basins are zero for this variable.
EMP-03  No data (meaning depends on the variable)
      applies to: all scopes; area_weighted and class_mixture; L6, L8
      template  : absent: The source dataset records no wetlands in {where}, so there is no wetland class. | missing: The source dataset has no {name} value for {where}{ (reason)}.
EMP-04  Not defined for an area (polity discharge)
      applies to: polity; dominant_basin (discharge rows)
      template  : {name} is not reported for a {area}: it describes a river network, not an

In [2]:
# Cell 3 — Fixtures: apply every rule to every Band A-E row; count where each applies and why not
fdf = A.apply_fixtures(fx, E)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of rows):')
print(A.reason_table(fdf).to_string())


res                  applied  not_applicable
id     scope                                
EMP-01 buffer              0             260
       polity              0             208
       single_basin      115            1081
EMP-02 buffer             17             243
       polity              6             202
       single_basin        0            1196
EMP-03 buffer              3             257
       polity              0             208
       single_basin        0            1196
EMP-04 buffer              0             260
       polity             12             196
       single_basin        0            1196
EMP-05 buffer             19             241
       polity              0             208
       single_basin        0            1196
EMP-06 buffer             55             205
       polity              0             208
       single_basin        0            1196

why a statement did not apply (count of rows):
id      reason                      
EMP-01  multi_u

In [4]:
# Cell 4 — Draft sentences on chosen fixtures (up to 2 per statement type), to read and judge
EXAMPLES = ['F01', 'F04', 'F18', 'F19', 'F20', 'F22']
for qid in EXAMPLES:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  EMP-01  Cropland extent is zero in this basin. No percentile is given: 43% of all L6 basins are zero for this variable.
  EMP-01  Karst area is zero in this basin. No percentile is given: 70% of all L6 basins are zero for this variable.

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  EMP-02  Karst area is zero across 100% of this buffer. No percentile is given: 70% of all L6 basins are zero for this variable.
  EMP-02  Karst area (upstream) is zero across 100% of this buffer. No percentile is given: 61% of all L6 basins are zero for this variable.
  EMP-06  No single percentile is given for Aridity index (P/PET) (upstream). Across the 9 basins in this buffer, the values fall into two groups: one centred near the 8th percentile (71% of the area with data) and one centred near the 37th percentile (29%); the centres are 29 percentile points apart.
  EMP-06  No single percentile is given for Cropland 

In [3]:
# Cell 5 — Population sample: how often each statement applies (share of all Band A-E rows), weighted and unweighted
pdf = A.apply_population(pop, W, E)
print('rows evaluated per statement type:', len(pdf) // len(E.STATEMENTS))
print('WEIGHTED by climate-zone size (percent of rows):')
print((A.rates(pdf) * 100).round(2).to_string())
print()
print('UNWEIGHTED (percent of rows):')
print((A.unweighted_rates(pdf) * 100).round(2).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts')


rows evaluated per statement type: 26260
WEIGHTED by climate-zone size (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
EMP-01       0.00      12.10      13.27
EMP-02       6.01       0.00       0.00
EMP-03       0.15       0.87       1.90
EMP-04       0.00       0.00       0.00
EMP-05       0.27       0.00       0.00
EMP-06      17.15       0.00       0.00

UNWEIGHTED (percent of rows):
kind    buffer L6  single L6  single L8
id                                     
EMP-01       0.00      14.12      14.81
EMP-02       8.22       0.00       0.00
EMP-03       2.16       3.68       4.85
EMP-04       0.00       0.00       0.00
EMP-05       0.70       0.00       0.00
EMP-06      15.02       0.00       0.00

polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts


In [6]:
# Cell 6 — Does every empty cell get exactly one statement? (a row is empty-ish if its status is not ok or its score is suppressed)
for name, df in (('fixtures', fdf), ('population', pdf)):
    g = df.groupby(['qid', 'variable']).agg(status=('status', 'first'), sup=('suppressed', 'first'), n=('applies', 'sum'))
    empty = g[(g.status != 'ok') | g.sup]
    ok = g[(g.status == 'ok') & ~g.sup]
    print(f"{name}: empty-ish rows {len(empty)} | covered by exactly one statement {int((empty.n == 1).sum())} | by none {int((empty.n == 0).sum())} | by several {int((empty.n > 1).sum())}")
    print(f"      ok rows that still get an Empty-cells statement (EMP-07 only expected): {int((ok.n > 0).sum())}, with more than one: {int((ok.n > 1).sum())}")
d = pdf[pdf.applies].assign(wa=lambda x: x.w)
comp = d.groupby(['kind', 'id']).wa.sum() / d.groupby('kind').wa.sum()
print()
print('weighted share of the rows that carry an Empty-cells statement, by cause (percent):')
print((comp.unstack('kind') * 100).round(1).to_string())


fixtures: empty-ish rows 227 | covered by exactly one statement 227 | by none 0 | by several 0
      ok rows that still get an Empty-cells statement (EMP-07 only expected): 11, with more than one: 0
population: empty-ish rows 5197 | covered by exactly one statement 5197 | by none 0 | by several 0
      ok rows that still get an Empty-cells statement (EMP-07 only expected): 102, with more than one: 0

weighted share of the rows that carry an Empty-cells statement, by cause (percent):
kind    buffer L6  single L6  single L8
id                                     
EMP-01        NaN       93.3       87.5
EMP-02       24.4        NaN        NaN
EMP-03        0.6        6.7       12.5
EMP-05        1.1        NaN        NaN
EMP-06       69.5        NaN        NaN
EMP-07        4.4        NaN        NaN


In [7]:
# Cell 7 — no_data: which variables, and whether the meaning table (provider pages, issue 5) covers them
nd = pdf[(pdf.id == 'EMP-03') & pdf.applies]
t = nd.groupby(['variable', 'kind']).size().unstack('kind', fill_value=0)
t['meaning'] = [E.no_data_meaning(v)[0] for v in t.index]
print(t.to_string())
print()
print('no_data variables with no entry in the meaning table (should be none):', sorted(set(nd.variable) - E.KNOWN_NO_DATA))


kind               buffer L6  single L6  single L8  meaning
variable                                                   
pct_clay                   7         53         23  missing
pct_clay_upstream          7         53         23  missing
pct_sand                   7         53         23  missing
pct_sand_upstream          7         53         23  missing
pct_silt                   7         53         23  missing
pct_silt_upstream          7         53         23  missing
slope_avg                 21         60         20  missing
slope_upstream            21         60         20  missing
stream_gradient           21         64         22  missing
wetland_class              6         84         52   absent

no_data variables with no entry in the meaning table (should be none): []


In [8]:
# Cell 8 — Numbers inside the statements: how far apart are the two groups (EMP-06), the guard threshold (EMP-07), and why low coverage (EMP-05)
print('EMP-06: distance between the two group centres, percentile points (weighted quantiles):')
print(A.stat_quantiles(pdf, ['EMP-06']).round(1).to_string(index=False))
g6 = pdf[(pdf.id == 'EMP-06') & pdf.applies].copy()
g6['d'] = g6.stat.astype(float)
GRID = [5, 10, 20, 30]
print('share of suppressed rows whose two centres are within X percentile points (weighted):')
print(pd.DataFrame({f'<= {x}': [float((g6.w * (g6.d <= x)).sum() / g6.w.sum())] for x in GRID}).round(3).to_string(index=False))
print()
gc = pd.DataFrame(E.guard_candidates(pop), columns=['qid', 'scope', 'variable', 'status', 'wz', 'adj'])
gc['w'] = gc.qid.map(W)
print('EMP-07: share of zero-aware buffer rows that would count as empty-in-effect at each threshold on weight_at_zero / coverage (weighted):')
TH = [0.8, 0.85, 0.9, 0.95, 1.0]
print(pd.DataFrame({f'>= {t}': [float((gc.w * (gc.adj >= t)).sum() / gc.w.sum())] for t in TH}).round(3).to_string(index=False))
print(f"  of which already outside_active_domain (engine): {float((gc.w * (gc.status == 'outside_active_domain')).sum() / gc.w.sum()):.3f}")
print()
lc = pd.DataFrame(E.low_coverage_causes(pop + fx), columns=['qid', 'variable', 'coverage', 'shortfall', 'rest'])
lc['cause'] = ['sea only' if r < 0.005 else 'basins without a value only' if s < 0.005 else 'both' for s, r in zip(lc.shortfall, lc.rest)]
print('EMP-05: low-coverage rows by cause (fixtures and population, unweighted):')
print(lc.cause.value_counts().to_string())


EMP-06: distance between the two group centres, percentile points (weighted quantiles):
    id      kind   n  p10  p25  p50  p75  p90
EMP-06 buffer L6 773  2.7  7.2 16.9 36.2 62.0
share of suppressed rows whose two centres are within X percentile points (weighted):
 <= 5  <= 10  <= 20  <= 30
0.187  0.328  0.567   0.71

EMP-07: share of zero-aware buffer rows that would count as empty-in-effect at each threshold on weight_at_zero / coverage (weighted):
 >= 0.8  >= 0.85  >= 0.9  >= 0.95  >= 1.0
  0.369    0.339   0.308    0.295   0.251
  of which already outside_active_domain (engine): 0.261

EMP-05: low-coverage rows by cause (fixtures and population, unweighted):
cause
sea only                       35
basins without a value only    14
both                            6


In [4]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Empty-cells rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
rows = []
for s in E.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Empty cells')
    row['fixture_result'] = f"{int(f.applied.sum())} rows applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of all A-E rows, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.2%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.2%}" for k in ru.columns)
                                + '; polities not sampled')
    row['sweep_result'] = {'EMP-06': 'no wording cutoff here; the call is the engine\'s (see Distribution family); distance between centres reported (Cell 8)',
                           'EMP-07': 'uses the engine\'s 0.90 threshold; sweep over 0.80 to 1.00 reported (Cell 8)'}.get(sid, 'no cutoff')
    rows.append(row)
n = P.upsert_master('Empty cells', rows)
print('master table now has', n, 'rows ->', P.MASTER)


master table now has 13 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
